# Step 1: Install Required Libraries

In [ ]:
# Install required libraries (use %pip for Jupyter compatibility)
%pip install openai ipywidgets python-dotenv --quiet

Enabling notebook extension jupyter-js-widgets/extension...
Paths used for configuration of notebook: 
    	/root/.jupyter/nbconfig/notebook.json
Paths used for configuration of notebook: 
    	
      - Validating: OK
Paths used for configuration of notebook: 
    	/root/.jupyter/nbconfig/notebook.json


# Step 2: Import Libraries

In [ ]:
import os
from IPython.display import display, Markdown, HTML, clear_output
import ipywidgets as widgets
from getpass import getpass
from openai import OpenAI
# AzureOpenAI import removed for simplicity

# Step 3: Configure Credentials

Credentials are loaded from environment variables. This is the production pattern — API keys are never hardcoded in source files.

Set the following variable before running:
- `OPENAI_API_KEY` — your OpenAI key (default provider)

This notebook falls back to `getpass` for interactive use. The production app (`app.py`) reads from environment only.

In [ ]:
# Load from environment variables (production method).
# Falls back to interactive prompt for notebook use only.
api_key = os.environ.get("OPENAI_API_KEY") or getpass("Enter your OpenAI API Key: ").strip()
print("Credentials loaded.")

🔐 Enter your Azure OpenAI API Key: ··········
🔗 Enter your Azure OpenAI Endpoint (e.g. https://your-resource.openai.azure.com): https://ai-dev-hub703422916896.cognitiveservices.azure.com/openai/deployments/finance-assistant-model/chat/completions?api-version=2025-01-01-preview


# Step 4: Connect to the API

The client is initialised for OpenAI. Switching providers in production is handled by environment variables, but this notebook is simplified for clarity.

In [ ]:
client = OpenAI(api_key=api_key)
model_id = os.environ.get("MODEL_NAME", "gpt-4o-mini")
print(f"Connected. Provider: OpenAI | Model: {model_id}")

# Step 5: Define the System Prompt

The system prompt enforces responsible AI constraints on every interaction. It is injected as the first message in the conversation history and never overwritten. This is the governance baseline for the assistant.

*This notebook version is the original prototype that inspired the full production app.*

In [ ]:
system_prompt = """You are a helpful and ethical AI-powered finance assistant.
You educate users on personal finance, savings, budgeting, loans, and investing.

You must ensure:
- Fairness: no biased advice or language
- Privacy: do not request or store personal data
- Transparency: state your limitations clearly
- Safety: avoid legal or personalised investment advice

You should:
- Use clear and inclusive language
- Recommend consulting qualified financial professionals for major decisions

You must apply these disclaimers in every response:
- You cannot provide legal, personalised investment, or tax advice
- Encourage users to consult financial professionals for critical decisions
- Avoid recommending specific financial products or institutions
"""


# Step 6: Initialise Conversation History

The system prompt is always the first entry. This list is passed to the API on every call so the model maintains context across the session (stateful, multi-turn conversation).

*This conversational pattern became the backbone of the full app's chat logic.*

In [ ]:
chat_history = [{"role": "system", "content": system_prompt}]

# Step 7: Define the GPT Interaction Function

The function appends the user message, calls the API with the full history, and appends the reply. A try/except block ensures the assistant degrades gracefully if the API is unavailable — the analytical summary is still returned rather than crashing.

*This error-handling and stateful chat loop was directly ported to the production codebase.*

In [ ]:
def ask_gpt(message: str) -> str:
    chat_history.append({"role": "user", "content": message})
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=chat_history,
            temperature=0.5,
            max_tokens=500,
        )
        reply = response.choices[0].message.content
    except Exception as exc:
        # Graceful degradation: returns a safe fallback instead of crashing.
        reply = (
            "The AI assistant is currently unavailable. "
            f"Please try again later. (Error: {exc})"
        )

    chat_history.append({"role": "assistant", "content": reply})
    return reply


# Step 8: Style the Widgets

*The focus on user experience and clarity in this prototype directly influenced the Streamlit UI in the final app.*

In [ ]:
display(HTML("""
<style>
    .widget-button { font-size: 16px !important; }
    textarea { font-family: 'Courier New', monospace; font-size: 15px; }
</style>
"""))


# Step 9: Create Input and Output Widgets

*This simple, interactive chat interface was the seed for the full-featured finance assistant.*

In [ ]:
input_box = widgets.Textarea(
    placeholder="Ask a finance question (max 500 characters)...",
    layout=widgets.Layout(width="100%", height="100px", border="1px solid #ddd", padding="10px"),
    style={"description_width": "initial"},
)
output_box = widgets.Output()
submit_button = widgets.Button(
    description="Ask",
    button_style="success",
    layout=widgets.Layout(width="100px"),
)
clear_button = widgets.Button(
    description="Clear",
    button_style="warning",
    layout=widgets.Layout(width="100px"),
)


# Step 10: Define Button Callbacks

Input is validated before reaching the API. Length is enforced at 500 characters. Clear resets both the UI and the conversation history so a new session begins.

*Robust input validation and session management were carried forward into the production app.*

In [ ]:
MAX_INPUT_LENGTH = 500

def on_submit(b):
    user_input = input_box.value.strip()
    if not user_input:
        return
    if len(user_input) > MAX_INPUT_LENGTH:
        with output_box:
            display(Markdown(
                f"**Input too long.** Please keep your question under {MAX_INPUT_LENGTH} characters."
            ))
        return

    reply = ask_gpt(user_input)
    with output_box:
        display(Markdown(
            f"---\n**You**\n\n{user_input}\n\n**Finance Assistant**\n\n{reply}\n"
        ))
    input_box.value = ""


def on_clear(b):
    global chat_history
    chat_history = [{"role": "system", "content": system_prompt}]
    output_box.clear_output()
    input_box.value = ""


# Step 11: Wire Buttons to Callbacks

*This modular, event-driven approach inspired the app's clean separation of UI and logic.*

In [ ]:
submit_button.on_click(on_submit)
clear_button.on_click(on_clear)

# Step 12: Launch the UI

*The notebook's interactive demo proved the concept and gave confidence to build a robust, production-ready finance intelligence system.*

In [ ]:
button_row = widgets.HBox([submit_button, clear_button])
ui_box = widgets.VBox([input_box, button_row, output_box])
display(ui_box)